# Stage 3 Polybot Backtest Visualization

Loads the Kotlin `polybot-backtest` outputs, summarizes contract-level results,
and visualizes the best contract in a stage2-style workflow.


In [80]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [81]:
BACKTEST_DIR = Path('../../data/polybot_backtest')

TRADES_DIR = Path('../../data/polygon_trades_processed')
SELECTED_WALLETS_PATH = Path('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

TOP_N_CONTRACTS = 10
AGGREGATE_RESAMPLE = '1h'
AGGREGATE_MAX_CONTRACTS = None  # Set int for faster iteration during exploration


## Load summaries

In [82]:
summaries_path = BACKTEST_DIR / 'summaries.json'
if not summaries_path.exists():
    raise FileNotFoundError(f'Missing summaries file: {summaries_path}')

summaries = pd.DataFrame(json.loads(summaries_path.read_text()))
if summaries.empty:
    raise ValueError('Backtest summaries are empty.')

summaries['firstTradeAt'] = pd.to_datetime(summaries['firstTradeAt'], utc=True)
summaries['lastTradeAt'] = pd.to_datetime(summaries['lastTradeAt'], utc=True)
summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).head(10)


,conditionId,eventCount,orderCount,fillCount,finalRealizedPnl,finalEstimatedPnl,totalImpliedPnl,firstTradeAt,lastTradeAt,copyWalletPnl
0,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,26809,2198,274,10.124284,-7.103478,-7.103356,2026-04-21 21:50:14+00:00,2026-04-25 06:27:50+00:00,-262.781362


In [83]:
totals = {
    'contracts_total': len(summaries),
    'contracts_with_fills': (summaries['fillCount'] > 0).sum(),
    'contracts_with_orders': (summaries['orderCount'] > 0).sum(),
    'total_estimated_pnl': summaries['finalEstimatedPnl'].sum(),
    'mean_estimated_pnl': summaries['finalEstimatedPnl'].mean(),
    'median_estimated_pnl': summaries['finalEstimatedPnl'].median(),
    'min_estimated_pnl': summaries['finalEstimatedPnl'].min(),
    'max_estimated_pnl': summaries['finalEstimatedPnl'].max(),
    'positive_pnl_contracts': (summaries['finalEstimatedPnl'] > 0).sum(),
    'negative_pnl_contracts': (summaries['finalEstimatedPnl'] < 0).sum(),
    'zero_pnl_contracts': (summaries['finalEstimatedPnl'] == 0).sum(),
    'total_orders': int(summaries['orderCount'].sum()),
    'total_fills': int(summaries['fillCount'].sum()),
    'total_events': int(summaries['eventCount'].sum()),
}

pd.DataFrame([totals]).T.rename(columns={0: 'value'})


,value
contracts_total,1.000000
contracts_with_fills,1.000000
contracts_with_orders,1.000000
total_estimated_pnl,-7.103478
mean_estimated_pnl,-7.103478
median_estimated_pnl,-7.103478
min_estimated_pnl,-7.103478
max_estimated_pnl,-7.103478
positive_pnl_contracts,0.000000
negative_pnl_contracts,1.000000


## Per-contract stats

In [84]:
display_cols = ['conditionId', 'finalEstimatedPnl', 'finalRealizedPnl', 'orderCount', 'fillCount', 'copyWalletPnl', 'firstTradeAt', 'lastTradeAt']

stats_sorted = summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).reset_index(drop=True)

# top N
stats_sorted.head(TOP_N_CONTRACTS)[display_cols]


,conditionId,finalEstimatedPnl,finalRealizedPnl,orderCount,fillCount,copyWalletPnl,firstTradeAt,lastTradeAt
0,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,-7.103478,10.124284,2198,274,-262.781362,2026-04-21 21:50:14+00:00,2026-04-25 06:27:50+00:00


## Aggregate PnL vs wallet cohort

In [85]:
selected_wallets_all = pd.read_parquet(SELECTED_WALLETS_PATH, columns=['wallet', 'wallet_group'])
if 'wallet_group' in selected_wallets_all.columns:
    watched_wallets = set(
        selected_wallets_all.loc[selected_wallets_all['wallet_group'] == 'predicting', 'wallet']
        .astype(str)
        .str.lower()
    )
    if not watched_wallets:
        watched_wallets = set(selected_wallets_all['wallet'].astype(str).str.lower())
else:
    watched_wallets = set(selected_wallets_all['wallet'].astype(str).str.lower())

wallet_parts = []
for shard in sorted(TRADES_DIR.glob('*.parquet')):
    df = pd.read_parquet(shard, columns=['wallet', 'dt', 'is_train', 'trade_pnl'])
    df['wallet'] = df['wallet'].astype(str).str.lower()
    mask = (~df['is_train']) & (df['wallet'].isin(watched_wallets))
    wallet_parts.append(df.loc[mask, ['dt', 'trade_pnl']])

if wallet_parts:
    wallet_df = pd.concat(wallet_parts, ignore_index=True)
    wallet_df['dt'] = pd.to_datetime(wallet_df['dt'], utc=True)
else:
    wallet_df = pd.DataFrame(columns=['dt', 'trade_pnl'])

wallet_hourly = (
    wallet_df.set_index('dt')['trade_pnl']
    .resample(AGGREGATE_RESAMPLE)
    .sum()
    .rename('wallet_trade_pnl')
    .to_frame()
) if not wallet_df.empty else pd.DataFrame(columns=['wallet_trade_pnl'])

contract_dirs = sorted((BACKTEST_DIR / 'contracts').glob('*'))
if AGGREGATE_MAX_CONTRACTS is not None:
    contract_dirs = contract_dirs[:AGGREGATE_MAX_CONTRACTS]

delta_parts = []
for contract_dir in contract_dirs:
    stats_path = contract_dir / 'stats.parquet'
    if not stats_path.exists():
        continue
    stats_df = pd.read_parquet(stats_path, columns=['dt', 'token_id', 'realized_pnl']).dropna(subset=['realized_pnl'])
    if stats_df.empty:
        continue
    stats_df['dt'] = pd.to_datetime(stats_df['dt'], utc=True)
    stats_df = stats_df.sort_values(['token_id', 'dt'])
    stats_df['delta_realized_pnl'] = stats_df.groupby('token_id')['realized_pnl'].diff().fillna(stats_df['realized_pnl'])
    delta_parts.append(stats_df[['dt', 'delta_realized_pnl']])

if delta_parts:
    polybot_delta = pd.concat(delta_parts, ignore_index=True)
    polybot_hourly = (
        polybot_delta.set_index('dt')['delta_realized_pnl']
        .resample(AGGREGATE_RESAMPLE)
        .sum()
        .rename('polybot_trade_pnl')
        .to_frame()
    )
else:
    polybot_hourly = pd.DataFrame(columns=['polybot_trade_pnl'])

combined_index = wallet_hourly.index.union(polybot_hourly.index).sort_values()
agg = pd.DataFrame(index=combined_index)
agg = agg.join(wallet_hourly, how='left').join(polybot_hourly, how='left')
agg[['wallet_trade_pnl', 'polybot_trade_pnl']] = agg[['wallet_trade_pnl', 'polybot_trade_pnl']].fillna(0.0)
agg['wallet_cum_trade_pnl'] = agg['wallet_trade_pnl'].cumsum()
agg['polybot_cum_trade_pnl'] = agg['polybot_trade_pnl'].cumsum()

aggregate_totals = pd.DataFrame([
    {'series': 'Wallet cohort cumulative trade_pnl', 'value': float(agg['wallet_cum_trade_pnl'].iloc[-1]) if not agg.empty else 0.0},
    {'series': 'Polybot cumulative traded pnl', 'value': float(agg['polybot_cum_trade_pnl'].iloc[-1]) if not agg.empty else 0.0},
    {'series': 'Polybot final estimated pnl (summaries)', 'value': float(summaries['finalEstimatedPnl'].sum())},
])
aggregate_totals


/private/tmp/ipykernel_91056/1421639643.py:46: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,series,value
0,Wallet cohort cumulative trade_pnl,158501.854055
1,Polybot cumulative traded pnl,18360.982906
2,Polybot final estimated pnl (summaries),-7.103478


In [86]:
fig_global = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=('Cumulative PnL (all watched wallets/contracts)', 'Per-bucket traded PnL')
)

fig_global.add_trace(
    go.Scatter(x=agg.index, y=agg['wallet_cum_trade_pnl'], mode='lines', name='Wallet cohort cumulative trade_pnl'),
    row=1, col=1
)
fig_global.add_trace(
    go.Scatter(x=agg.index, y=agg['polybot_cum_trade_pnl'], mode='lines', name='Polybot cumulative traded pnl'),
    row=1, col=1
)

fig_global.add_trace(
    go.Bar(x=agg.index, y=agg['wallet_trade_pnl'], name='Wallet cohort trade_pnl (bucket)'),
    row=2, col=1
)
fig_global.add_trace(
    go.Bar(x=agg.index, y=agg['polybot_trade_pnl'], name='Polybot traded pnl (bucket)'),
    row=2, col=1
)

fig_global.update_layout(
    height=850, template='plotly_white',
    title='Global aggregation: all watched wallets/contracts vs polybot traded pnl',
    barmode='group',
)
fig_global.show()


## Order-fill delay analysis

In [87]:
# Load fill data for ALL contracts — computes delays and wallet stats in one pass
all_delays = []
wallet_parts = []

contract_dirs = sorted((BACKTEST_DIR / 'contracts').glob('*'))
if AGGREGATE_MAX_CONTRACTS is not None:
    contract_dirs = contract_dirs[:AGGREGATE_MAX_CONTRACTS]

for contract_dir in contract_dirs:
    fills_path = contract_dir / 'fills.parquet'
    if not fills_path.exists():
        continue
    df = pd.read_parquet(fills_path, columns=[
        'condition_id', 'token_id', 'dt',
        'trigger_wallet', 'synthetic_side',
        'synthetic_price', 'fill_quantity',
        'order_timestamp', 'fill_timestamp'
    ])
    if df.empty:
        continue
        df['dt'] = pd.to_datetime(df['dt'], utc=True)
    df['order_timestamp'] = pd.to_datetime(df['order_timestamp'], utc=True)
    df['fill_timestamp'] = pd.to_datetime(df['fill_timestamp'], utc=True)
    df['delay_s'] = (df['fill_timestamp'] - df['order_timestamp']).dt.total_seconds()
    df['notional'] = df['fill_quantity'] * df['synthetic_price']
    all_delays.append(df[['condition_id', 'delay_s']])
    wallet_parts.append(df[[
        'condition_id', 'token_id', 'dt', 'trigger_wallet',
        'synthetic_side', 'synthetic_price', 'fill_quantity',
        'notional', 'delay_s'
    ]])

all_delays_df = pd.concat(all_delays, ignore_index=True) if all_delays else pd.DataFrame(columns=['condition_id', 'delay_s'])
all_delays_df = all_delays_df[all_delays_df['delay_s'] >= 0]

wallet_all = pd.concat(wallet_parts, ignore_index=True) if wallet_parts else pd.DataFrame(
    columns=['condition_id', 'token_id', 'dt', 'trigger_wallet', 'synthetic_side', 'synthetic_price', 'fill_quantity', 'notional', 'delay_s']
)

print(f'Fill events loaded: {len(all_delays_df):,}')
print(f'Contracts with fills: {all_delays_df["condition_id"].nunique():,}')
print(f'Unique trigger wallets: {wallet_all["trigger_wallet"].nunique()}')


/private/tmp/ipykernel_91056/949071710.py:23: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/private/tmp/ipykernel_91056/949071710.py:22: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



Fill events loaded: 39,602
Contracts with fills: 692
Unique trigger wallets: 51


In [88]:
# -- ALL contracts --
delay_stats = all_delays_df['delay_s'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
print('Order-fill delay stats — ALL contracts with fills')
print(delay_stats.to_string())
print()

# -- TOP-N by abs PnL --
top_ids = set(summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).head(TOP_N_CONTRACTS)['conditionId'])
top_delays = all_delays_df[all_delays_df['condition_id'].isin(top_ids)]

if not top_delays.empty:
    print(f'Order-fill delay stats — TOP-{TOP_N_CONTRACTS} contracts by abs PnL')
    print(f'Fill events: {len(top_delays):,} | Contracts: {top_delays["condition_id"].nunique()}')
    print(top_delays['delay_s'].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_string())
else:
    print(f'No fill events found in the top-{TOP_N_CONTRACTS} contracts.')


Order-fill delay stats — ALL contracts with fills
count     39602.000000
mean        215.845614
std        2354.561655
min           1.000000
25%           4.000000
50%          16.000000
75%          65.000000
90%         233.000000
95%         500.000000
99%        2879.970000
max      222478.000000

Order-fill delay stats — TOP-10 contracts by abs PnL
Fill events: 274 | Contracts: 1
count    274.000000
mean      21.014599
std       74.103826
min        2.000000
25%        2.000000
50%        2.000000
75%        8.000000
90%       25.400000
95%       91.700000
99%      488.540000
max      684.000000


In [89]:
import numpy as np

log_delays = np.log10(all_delays_df['delay_s'].clip(lower=1))

fig_delay = go.Figure()
fig_delay.add_trace(go.Histogram(
    x=log_delays,
    nbinsx=80,
    name='All contracts'
))

median_val = all_delays_df['delay_s'].median()
median_log = np.log10(max(median_val, 1))
fig_delay.add_vline(x=median_log, line_dash='dash', line_color='red',
                    annotation_text=f'median={median_val:.0f}s', annotation_position='top right')

max_log = int(np.ceil(log_delays.max()))
tickvals = list(range(0, max_log + 1))
ticktext = ['1s', '10s', '1m', '10m', '1.7h', '11.6h', '4.8d'][:max_log + 1]

fig_delay.update_layout(
    title='Order-fill delay distribution (all contracts) — log10 scale',
    xaxis_title='Delay',
    yaxis_title='Fill events',
    template='plotly_white', height=500,
)
fig_delay.update_xaxes(tickvals=tickvals, ticktext=ticktext)
fig_delay.show()


## Wallet-level stats

In [90]:
from collections import deque

# Aggregate wallet stats
wallet_stats = wallet_all.groupby('trigger_wallet').agg(
    fill_count=('condition_id', 'count'),
    unique_contracts=('condition_id', 'nunique'),
    total_notional=('notional', 'sum'),
    avg_notional=('notional', 'mean'),
    median_delay_s=('delay_s', 'median'),
    avg_delay_s=('delay_s', 'mean'),
).sort_values('fill_count', ascending=False)

# FIFO PnL attribution per (condition_id, token_id)
# BUY fills go into a queue; SELL fills match against the queue
# PnL = (sell_price - buy_price) * matched_qty, attributed to SELL's trigger_wallet
pnl_per_wallet = {}

for (cond_id, token_id), grp in wallet_all.groupby(['condition_id', 'token_id'], sort=False):
    grp = grp.sort_values('dt')
    buy_queue = deque()

    for _, row in grp.iterrows():
        side = row['synthetic_side']
        qty = row['fill_quantity']
        price = row['synthetic_price']
        wallet = row['trigger_wallet'] if pd.notna(row['trigger_wallet']) else None

        if side == 'BUY':
            buy_queue.append([qty, price, wallet])
        else:
            remaining = qty
            while remaining > 1e-10 and buy_queue:
                buy_qty, buy_price, _ = buy_queue[0]
                matched = min(remaining, buy_qty)
                pnl = (price - buy_price) * matched
                if wallet is not None:
                    pnl_per_wallet[wallet] = pnl_per_wallet.get(wallet, 0.0) + pnl
                if matched >= buy_qty - 1e-10:
                    buy_queue.popleft()
                else:
                    buy_queue[0][0] -= matched
                remaining -= matched

wallet_stats['attributed_pnl'] = wallet_stats.index.to_series().map(pnl_per_wallet).fillna(0.0).round(2)

wallet_stats['total_notional'] = wallet_stats['total_notional'].round(2)
wallet_stats['avg_notional'] = wallet_stats['avg_notional'].round(2)
wallet_stats['median_delay_s'] = wallet_stats['median_delay_s'].round(1)
wallet_stats['avg_delay_s'] = wallet_stats['avg_delay_s'].round(1)

print('Top 15 wallets by attributed PnL:')
wallet_stats = wallet_stats.sort_values('attributed_pnl', ascending=False, key=abs)
wallet_stats.head(15)


Top 15 wallets by attributed PnL:


,fill_count,unique_contracts,total_notional,avg_notional,median_delay_s,avg_delay_s,attributed_pnl
trigger_wallet,,,,,,,
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,1081,62,103217.18,95.48,11.0,77.8,-5680.79
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,3513,119,216798.84,61.71,4.0,28.2,-4545.97
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,745,70,21187.04,28.44,11.0,180.7,-3275.65
0x22dbd51e1a4e10fff072fa24300238c64033190f,801,66,35094.87,43.81,13.0,171.4,1979.35
0x39d0f1dca6fb7e5514858c1a337724a426764fe8,1256,41,32295.60,25.71,14.0,43.6,1563.27
0x41583f2efc720b8e2682750fffb67f2806fece9f,789,62,18296.18,23.19,12.0,62.0,-1553.07
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,332,20,25419.88,76.57,9.0,25.7,-1525.95
0x927ffeba73eb63ffb7e576aca7d32f78e54be7e7,91,18,4333.45,47.62,37.0,188.5,896.54
0x40471b34671887546013ceb58740625c2efe7293,108,14,4926.91,45.62,35.0,75.9,-863.64


In [91]:
side_pivot = wallet_all.groupby(['trigger_wallet', 'synthetic_side']).size().unstack(fill_value=0)
side_pivot['total'] = side_pivot.sum(axis=1)
side_pivot = side_pivot.sort_values('total', ascending=False)

print('Top 15 wallets by side distribution:')
side_pivot.head(15)


Top 15 wallets by side distribution:


synthetic_side,SELL,total
trigger_wallet,,
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,3513,3513
0x39d0f1dca6fb7e5514858c1a337724a426764fe8,1256,1256
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,1081,1081
0x22dbd51e1a4e10fff072fa24300238c64033190f,801,801
0x41583f2efc720b8e2682750fffb67f2806fece9f,789,789
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,745,745
0x92d0cb81e6c891b835c8e0272e8c323095bd269e,524,524
0x0ad7f3411bc87f0b5362155e638f04ef05700971,469,469
0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e,417,417


In [92]:
top15 = wallet_stats.head(15)

fig_wallet = go.Figure()
fig_wallet.add_trace(go.Bar(
    x=top15.index.str[:10] + '...',
    y=top15['fill_count'],
    marker_color='royalblue',
    text=top15['fill_count'],
    textposition='outside',
))

fig_wallet.update_layout(
    title='Top wallets by fill count',
    xaxis_title='Wallet',
    yaxis_title='Fill count',
    template='plotly_white',
    height=500,
    margin=dict(b=120),
)
fig_wallet.update_xaxes(tickangle=45)
fig_wallet.show()


In [93]:
top15_pnl = wallet_stats.head(15)

fig_pnl = go.Figure()
colors = ['green' if v >= 0 else 'red' for v in top15_pnl['attributed_pnl']]
fig_pnl.add_trace(go.Bar(
    x=top15_pnl.index.str[:10] + '...',
    y=top15_pnl['attributed_pnl'],
    marker_color=colors,
    text=top15_pnl['attributed_pnl'].round(1),
    textposition='outside',
))

fig_pnl.update_layout(
    title='Top wallets by attributed PnL (FIFO)',
    xaxis_title='Wallet',
    yaxis_title='Attributed PnL',
    template='plotly_white',
    height=500,
    margin=dict(b=120),
)
fig_pnl.update_xaxes(tickangle=45)
fig_pnl.show()


## Top contracts PnL comparison

In [94]:
top_contracts = summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).head(TOP_N_CONTRACTS).reset_index(drop=True)

pnl_parts = []
for _, row in top_contracts.iterrows():
    cid = row['conditionId']
    stats_path = BACKTEST_DIR / 'contracts' / cid / 'stats.parquet'
    if not stats_path.exists():
        continue
    sdf = pd.read_parquet(stats_path, columns=['dt', 'token_id', 'realized_pnl']).dropna(subset=['realized_pnl'])
    if sdf.empty:
        continue
    sdf['dt'] = pd.to_datetime(sdf['dt'], utc=True)
    sdf = sdf.sort_values(['token_id', 'dt'])
    sdf['delta'] = sdf.groupby('token_id')['realized_pnl'].diff().fillna(sdf['realized_pnl'])
    contract_pnl = sdf.groupby('dt', as_index=False)['delta'].sum()
    contract_pnl['cum_pnl'] = contract_pnl['delta'].cumsum()
    contract_pnl['condition_id'] = cid
    pnl_parts.append(contract_pnl[['dt', 'cum_pnl', 'condition_id']])

if pnl_parts:
    pnl_all = pd.concat(pnl_parts, ignore_index=True)

    fig_top = go.Figure()
    for cid, grp in pnl_all.groupby('condition_id'):
        short_id = cid[:10] + '...' + cid[-6:]
        fig_top.add_trace(go.Scatter(
            x=grp['dt'], y=grp['cum_pnl'],
            mode='lines', name=short_id
        ))

    fig_top.update_layout(
        title=f'Cumulative PnL for top-{TOP_N_CONTRACTS} contracts by absolute PnL',
        xaxis_title='Date', yaxis_title='Cumulative PnL',
        template='plotly_white', height=600,
        legend=dict(font=dict(size=8)),
    )
    fig_top.show()
else:
    print('No stats.parquet data found for top contracts.')


## Detailed timeline: best contract

In [95]:
best_cid = summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).iloc[0]['conditionId']
best_dir = BACKTEST_DIR / 'contracts' / best_cid

parts = []
for row_type, file_name in [('stats', 'stats.parquet'), ('order', 'orders.parquet'), ('fill', 'fills.parquet'), ('public_trade', 'public_trades.parquet'), ('plan_fact', 'plan_facts.parquet')]:
    path = best_dir / file_name
    if path.exists():
        df = pd.read_parquet(path)
        df['row_type'] = row_type
        parts.append(df)

if not parts:
    raise FileNotFoundError(f'No parquet files found in {best_dir}')

events = pd.concat(parts, ignore_index=True)
events['dt'] = pd.to_datetime(events['dt'], utc=True)

print(f'Best contract: {best_cid}')
summary = json.loads((best_dir / 'summary.json').read_text())
print(f'Estimated PnL: {summary["finalEstimatedPnl"]:.2f}')
events.head()


Best contract: 0x084a4aac9da433824013ad192196193d7d1bd8293c0fedc9eb7e687909fd1d0f
Estimated PnL: -7.10


,condition_id,token_id,outcome,dt,position_after,total_buy_volume,realized_pnl,est_pnl,best_bid,best_ask,...,fill_quantity,implied_pnl,order_timestamp,accepted_at,fill_timestamp,wallet,public_side,public_price,public_quantity,facts
0,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,Yes,2026-04-21 23:13:16+00:00,62.0,4.34,0.0,0.306006,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,Yes,2026-04-21 23:17:22+00:00,124.0,8.68,0.0,0.583476,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,Yes,2026-04-21 23:17:24+00:00,324.0,22.68,0.0,1.451773,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,Yes,2026-04-21 23:18:10+00:00,699.0,45.18,0.0,2.299150,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,0x084a4aac9da433824013ad192196193d7d1bd8293c0f...,No,2026-04-21 23:18:10+00:00,0.0,0.00,0.0,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [96]:
timeline = events[events['row_type'] == 'stats'][['dt', 'token_id', 'outcome', 'position_after', 'realized_pnl', 'est_pnl', 'best_bid', 'best_ask', 'fair_price', 'vwap5m', 'vwap15m', 'vwap1h']].copy()
timeline = timeline.sort_values(['dt', 'token_id']).reset_index(drop=True)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     subplot_titles=('Estimated / Realized PnL', 'Position', 'Prices'))

for token_id, grp in timeline.groupby('token_id', sort=False):
    label = grp['outcome'].iloc[0]
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['est_pnl'], name=f'{label} est pnl', mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['realized_pnl'], name=f'{label} realized pnl', mode='lines', line=dict(dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['position_after'], name=f'{label} position', mode='lines'), row=2, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['fair_price'], name=f'{label} fair', mode='lines'), row=3, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['best_bid'], name=f'{label} bid', mode='lines', line=dict(dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['best_ask'], name=f'{label} ask', mode='lines', line=dict(dash='dot')), row=3, col=1)

fig.update_layout(height=1000, template='plotly_white', title=f'Detailed timeline: {best_cid[:10]}...{best_cid[-6:]}')
fig.show()
